Build a SIMULATED loan book, priced off REAL Bank of Canada rates.

Run the cells in order (or "Run All"). Needs data/rates.csv - run
`python src\fetch_rates.py` first.
Output: data/loans.csv - one row per loan.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import yaml

config = yaml.safe_load(open("config/pipeline.yml"))
portfolio = config["portfolio"]
risk = config["risk"]
window = config["window"]

rng = np.random.default_rng(config["seed"])  # seeded -> same loans every run
n = portfolio["n_loans"]

rates = pd.read_csv("data/rates.csv", parse_dates=["month"])
first_month = rates["month"].min()
last_month_num = len(rates) - 1  # rates has one row per month, so row number = month number

# A calendar of month start dates, long enough to name renewals up to 10 years out
calendar = pd.date_range(first_month, periods=len(rates) + 120, freq="MS")


def to_month_num(date_text: str) -> int:
    """Months since the first month of rate data (Jan 2015 = 0)."""
    date = pd.Timestamp(date_text)
    return (date.year - first_month.year) * 12 + (date.month - first_month.month)


print(f"{n:,} loans to simulate; rates cover {len(rates)} months")

## 2. Who the borrowers are

In [ ]:
loans = pd.DataFrame({"loan_id": np.arange(1, n + 1)})

mix = portfolio["product_mix"]
loans["product"] = rng.choice(list(mix), size=n, p=list(mix.values()))

shares = portfolio["regional_shares"]
loans["province"] = rng.choice(list(shares), size=n, p=list(shares.values()))

score = portfolio["credit_score"]
loans["credit_score"] = rng.normal(score["mean"], score["sd"], n).round().clip(300, 900).astype(int)

# Fixed or variable rate: mortgages are a mix, HELOCs float with prime
is_mortgage = loans["product"] == "mortgage"
loans["rate_type"] = "fixed"
loans.loc[loans["product"] == "heloc", "rate_type"] = "variable"
fixed_share = portfolio["mortgage_fixed_share"]
loans.loc[is_mortgage, "rate_type"] = rng.choice(
    ["fixed", "variable"], size=is_mortgage.sum(), p=[fixed_share, 1 - fixed_share]
)

# Loan size: uniform between each product's min and max
loans["original_balance"] = 0.0
for product, (low, high) in portfolio["original_balance"].items():
    is_product = loans["product"] == product
    loans.loc[is_product, "original_balance"] = rng.uniform(low, high, is_product.sum()).round(2)

# Signing month: any month from origination_start to the end of our data.
# Auto loans are paid off after 72 months, so they must start late enough
# to still be open when the performance window begins.
earliest = to_month_num(window["origination_start"])
latest = to_month_num(window["performance_end"])
loans["orig_month_num"] = rng.integers(earliest, latest + 1, n)
is_auto = loans["product"] == "auto"
auto_earliest = to_month_num(window["performance_start"]) - risk["auto"]["term_months"] + 1
loans.loc[is_auto, "orig_month_num"] = rng.integers(auto_earliest, latest + 1, is_auto.sum())
loans["origination_month"] = calendar[loans["orig_month_num"]]

loans.head()

## 3. Interest rate at signing, from the real rates that month

In [ ]:
goc_5yr = rates["goc_5yr_yield"].to_numpy()  # index by month number
prime = rates["prime_rate"].to_numpy()
goc_at_signing = goc_5yr[loans["orig_month_num"]]
prime_at_signing = prime[loans["orig_month_num"]]

# Credit score effect: +1 for every 100 points above the reference score.
# Each product prices this differently (trap #4 in the brief).
score_vs_reference = (loans["credit_score"] - score["reference"]) / 100

m, h, a, c = risk["mortgage"], risk["heloc"], risk["auto"], risk["card"]
fixed_mortgage = np.maximum(
    goc_at_signing + m["fixed_spread_over_goc_5yr"] - m["score_sensitivity_per_100pts"] * score_vs_reference,
    m["fixed_rate_floor"],
)
variable_mortgage = np.maximum(
    prime_at_signing + m["variable_spread_over_prime"] - m["score_sensitivity_per_100pts"] * score_vs_reference,
    m["variable_rate_floor"],
)
heloc = prime_at_signing + h["spread_over_prime"] - h["score_sensitivity_per_100pts"] * score_vs_reference
auto = goc_at_signing + a["spread_over_goc_5yr"] - a["score_sensitivity_per_100pts"] * score_vs_reference
card = np.clip(c["base_apr"] - c["score_sensitivity_per_100pts"] * score_vs_reference, *c["apr_range"])

is_fixed_mortgage = is_mortgage & (loans["rate_type"] == "fixed")
is_variable_mortgage = is_mortgage & (loans["rate_type"] == "variable")
loans["rate_at_signing"] = np.select(
    [is_fixed_mortgage, is_variable_mortgage, loans["product"] == "heloc", is_auto, loans["product"] == "card"],
    [fixed_mortgage, variable_mortgage, heloc, auto, card],
).round(2)

loans.groupby("product")["rate_at_signing"].describe().round(2)

## 4. Fixed-mortgage renewals in the window, and the payment shock

In [ ]:
# Only FIXED mortgages get a renewal shock: variable rates already move
# with prime every month, so there's no single repricing event.


def monthly_payment(balance, annual_rate_pct, months):
    """Standard mortgage payment formula."""
    r = annual_rate_pct / 100 / 12
    return balance * r / (1 - (1 + r) ** -months)


def balance_after(balance, annual_rate_pct, payment, months_paid):
    """What's still owed after making `months_paid` payments."""
    r = annual_rate_pct / 100 / 12
    growth = (1 + r) ** months_paid
    return balance * growth - payment * (growth - 1) / r


term = m["term_months"]
amortization = m["amortization_months"]
window_start = to_month_num(window["renewal_window_start"])
window_end = to_month_num(window["renewal_window_end"])
orig = loans["orig_month_num"].to_numpy()

# The first renewal on or after the window start. A loan signed in 2016
# renewed in 2021 and renews again in 2026 - that 2026 renewal is the one we want.
terms_completed = np.maximum(np.ceil((window_start - orig) / term), 1).astype(int)
renewal_num = orig + term * terms_completed
term_start_num = renewal_num - term  # when the current rate was locked in
renews = (is_fixed_mortgage & (renewal_num <= window_end)).to_numpy()

# Old rate: locked in at the start of the current term.
# New rate: rate at renewal, or the last real month if renewal is in the future (held flat).
renewal_rate_month = np.minimum(renewal_num, last_month_num)
mortgage_score_adj = m["score_sensitivity_per_100pts"] * score_vs_reference
old_rate = np.maximum(goc_5yr[term_start_num] + m["fixed_spread_over_goc_5yr"] - mortgage_score_adj, m["fixed_rate_floor"])
new_rate = np.maximum(goc_5yr[renewal_rate_month] + m["fixed_spread_over_goc_5yr"] - mortgage_score_adj, m["fixed_rate_floor"])

# Simplification: we amortize at the current term's rate since signing. For a
# loan on its second term, the first term's rate differed a little - this
# slightly misstates the balance at renewal, not the size of the rate jump.
months_paid = renewal_num - orig
old_payment = monthly_payment(loans["original_balance"], old_rate, amortization)
balance_at_renewal = balance_after(loans["original_balance"], old_rate, old_payment, months_paid)
new_payment = monthly_payment(balance_at_renewal, new_rate, amortization - months_paid)

loans["renews_in_window"] = renews
loans["renewal_month"] = pd.Series(calendar[renewal_num]).where(renews)
loans["old_rate"] = np.where(renews, old_rate, np.nan).round(2)
loans["new_rate"] = np.where(renews, new_rate, np.nan).round(2)
loans["balance_at_renewal"] = np.where(renews, balance_at_renewal, np.nan).round(2)
loans["old_payment"] = np.where(renews, old_payment, np.nan).round(2)
loans["new_payment"] = np.where(renews, new_payment, np.nan).round(2)
loans["payment_increase"] = (loans["new_payment"] - loans["old_payment"]).round(2)
loans["payment_shock_pct"] = (100 * (loans["new_payment"] / loans["old_payment"] - 1)).round(2)
loans["new_rate_source"] = np.where(
    ~renews, None,
    np.where(renewal_num <= last_month_num, "real rate", "assumed: last real rate held flat"),
)

print(f"{renews.sum():,} fixed mortgages renew between {window['renewal_window_start']} and {window['renewal_window_end']}")
loans[renews].head()

## 5. Save

In [ ]:
loans["source"] = "SIMULATED - not real customers"
loans = loans.drop(columns=["orig_month_num"])
loans.to_csv("data/loans.csv", index=False)
print(f"Saved data/loans.csv: {len(loans):,} rows, {len(loans.columns)} columns")

## 6. Sanity checks - look at these, don't just trust them

In [ ]:
print("Loans by product:")
print(loans["product"].value_counts(normalize=True).round(3))

renewing = loans[loans["renews_in_window"]].copy()
renewing["signed_year"] = renewing["renewal_month"].dt.year - term // 12

print("\nRenewing fixed mortgages, by year the current term started:")
print(
    renewing.groupby("signed_year").agg(
        loans=("loan_id", "count"),
        median_old_rate=("old_rate", "median"),
        median_new_rate=("new_rate", "median"),
        median_shock_pct=("payment_shock_pct", "median"),
    ).round(2)
)

# The brief says some loans should get CHEAPER at renewal (signed when rates were high)
cheaper = (renewing["payment_increase"] < 0).mean()
print(f"\nShare of renewals where the payment goes DOWN: {cheaper:.1%}")